In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.functions import udf

In [3]:
spark = SparkSession.builder \
    .appName("retail analytics") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 06:07:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)

orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)

products = spark.read.csv("data/products.csv", header=True, inferSchema=True)

returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

26/06/16 06:07:38 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [5]:
customers.show(5)
orders.show(5)
order_items.show(5)
products.show(5)
returns.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|payment_mode|order_status|
+--------+-----------+----------+------------+------------+
|       1|      54630|2024-01-25| Credit Card|   Delivered|
|       2|      22415|2024-07-08|

In [6]:
products.head()


Row(product_id=1, product_name='Product_1', category='Home & Kitchen', brand='Brand_A', unit_cost=509.39)

In [7]:
#qn 2
products.createOrReplaceTempView("products")
order_items.createOrReplaceTempView("order_items")

In [8]:
category_sales = spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""")

category_sales.show()

(category_sales.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/total_sales_by_category"))



+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|        Beauty|7.626693058999963E8|
|Home & Kitchen|7.581388732799902E8|
|         Books|7.464907783499908E8|
|          Toys|7.446190722999846E8|
|   Electronics|7.442665041099958E8|
|        Sports|7.433388681300008E8|
|      Clothing|7.419227945699946E8|
+--------------+-------------------+



In [9]:
#qn 3
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

In [10]:
top_customers = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS total_spent
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_spent DESC
LIMIT 5
""")

top_customers.show()

(category_sales.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/top_10_customers"))

+-----------+--------------+------------------+
|customer_id| customer_name|       total_spent|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|169060.39999999997|
|      23289|Customer_23289|161573.80000000002|
|      52275|Customer_52275|153364.78999999998|
|      61218|Customer_61218|         153067.55|
+-----------+--------------+------------------+



In [11]:
#qn 4
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

In [12]:
monthly_sales = spark.sql("""
SELECT
    DATE_FORMAT(o.order_date, 'yyyy-MM') AS month,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY DATE_FORMAT(o.order_date, 'yyyy-MM')
ORDER BY month
""")

monthly_sales.show()


(category_sales.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/monthsales"))

+-------+--------------------+
|  month|         total_sales|
+-------+--------------------+
|2024-01| 4.445777757600032E8|
|2024-02| 4.153661441999996E8|
|2024-03| 4.436282454099993E8|
|2024-04|4.2782097433999574E8|
|2024-05|4.4481061894999886E8|
|2024-06| 4.317051540600023E8|
|2024-07| 4.436705191199994E8|
|2024-08|4.4109517702000153E8|
|2024-09| 4.310715260799986E8|
|2024-10|4.4136378931000113E8|
|2024-11| 4.336233640400014E8|
|2024-12| 4.427129083499967E8|
+-------+--------------------+



In [13]:
#qn 5 Calculate the return rate for each product category.
products.createOrReplaceTempView("products")
order_items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")

In [14]:
return_rate = spark.sql("""
SELECT
    p.category,
    COUNT(r.return_id) AS total_returns,
    COUNT(oi.order_item_id) AS total_sales,
    ROUND(
        COUNT(r.return_id) * 100.0 /
        COUNT(oi.order_item_id),
        2
    ) AS return_rate_percentage
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN returns r
    ON oi.order_id = r.order_id
GROUP BY p.category
ORDER BY return_rate_percentage DESC
""")

return_rate.show()
return_rate.write \
 .mode("overwrite") \
 .option("header", "true") \
 .csv("output/return_rate_for_each_product_category.")

+--------------+-------------+-----------+----------------------+
|      category|total_returns|total_sales|return_rate_percentage|
+--------------+-------------+-----------+----------------------+
|          Toys|        43382|     430418|                 10.08|
|        Beauty|        43194|     430547|                 10.03|
|        Sports|        42530|     424412|                 10.02|
|         Books|        42809|     427086|                 10.02|
|Home & Kitchen|        43418|     434034|                 10.00|
|   Electronics|        42601|     425896|                 10.00|
|      Clothing|        42660|     427607|                  9.98|
+--------------+-------------+-----------+----------------------+



In [15]:
#qn 6
avg_order_value = spark.sql("""
SELECT
    ROUND(
        SUM(quantity * selling_price) /
        COUNT(DISTINCT order_id),
        2
    ) AS average_order_value
FROM order_items
""")

avg_order_value.show()

[Stage 83:>                                                         (0 + 2) / 2]

+-------------------+
|average_order_value|
+-------------------+
|            5517.29|
+-------------------+



In [16]:
avg_order_value.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/question6_avg_order_value")

In [17]:
top_products = spark.sql("""
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(oi.quantity) AS total_quantity_sold
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
ORDER BY total_quantity_sold DESC
LIMIT 10
""")

top_products.show()

[Stage 94:=============================>                            (1 + 1) / 2]

+----------+-------------+--------------+-------------------+
|product_id| product_name|      category|total_quantity_sold|
+----------+-------------+--------------+-------------------+
|     46817|Product_46817|        Sports|                287|
|     48617|Product_48617|Home & Kitchen|                285|
|     15217|Product_15217|          Toys|                283|
|     38170|Product_38170|   Electronics|                281|
|     28034|Product_28034|        Sports|                280|
|     19230|Product_19230|         Books|                279|
|     24689|Product_24689|        Sports|                275|
|     41174|Product_41174|   Electronics|                275|
|     22942|Product_22942|        Sports|                274|
|     10971|Product_10971|      Clothing|                273|
+----------+-------------+--------------+-------------------+



In [18]:
top_products.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/question7_top_products")

In [19]:
#multi_category_customers.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/question8_multi_category_customers")S
multi_category_customers = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(DISTINCT p.category) AS category_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    c.customer_id,
    c.customer_name
HAVING COUNT(DISTINCT p.category) > 1
ORDER BY category_count DESC
""")

multi_category_customers.show()

26/06/16 06:29:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:29:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 112:============================>                            (1 + 1) / 2]

+-----------+--------------+--------------+
|customer_id| customer_name|category_count|
+-----------+--------------+--------------+
|      14717|Customer_14717|             7|
|      41194|Customer_41194|             7|
|      75263|Customer_75263|             7|
|      24067|Customer_24067|             7|
|      64392|Customer_64392|             7|
|      29035|Customer_29035|             7|
|      10446|Customer_10446|             7|
|      96385|Customer_96385|             7|
|      95950|Customer_95950|             7|
|      42661|Customer_42661|             7|
|      47624|Customer_47624|             7|
|      31859|Customer_31859|             7|
|      77624|Customer_77624|             7|
|       7222| Customer_7222|             7|
|      75072|Customer_75072|             7|
|      42613|Customer_42613|             7|
|      67234|Customer_67234|             7|
|      60672|Customer_60672|             7|
|      43698|Customer_43698|             7|
|      84622|Customer_84622|    

In [20]:
multi_category_customers.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/question8_multi_category_customers")

26/06/16 06:29:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:29:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                